## Ingress

In this notebook, we look at a Kubernetes Ingress.

- Make sure you have a Kubernetes cluster (Docker Desktop) running.
- Also make sure you have installed the `kubectl` tool on your computer.

## Install the Nginx Ingress Controller

- Docker Desktop's Kubernetes cluster does not come with an Ingress Controller pre-installed.
- We will use `Helm` to install the Nginx Ingress Controller.

In [ ]:
# Add ingress-nginx to the local Helm repository
!helm repo rm ingress-nginx
!helm repo add ingress-nginx https://kubernetes.github.io/ingress-nginx
!helm repo update
!helm repo ls

# Deploy the Helm Chart for ingress-nginx to Docker Desktop's Kubernetes cluster
!helm uninstall ingress-nginx --namespace ingress-nginx
!helm upgrade --install ingress-nginx ingress-nginx/ingress-nginx --create-namespace --namespace ingress-nginx
!helm list -A

"ingress-nginx" has been removed from your repositories
"ingress-nginx" has been added to your repositories
Hang tight while we grab the latest from your chart repositories...
...Successfully got an update from the "metrics-server" chart repository
...Successfully got an update from the "kubernetes-dashboard" chart repository
...Successfully got an update from the "ingress-nginx" chart repository
Update Complete. ⎈Happy Helming!⎈
NAME                	URL                                              
metrics-server      	https://kubernetes-sigs.github.io/metrics-server/
kubernetes-dashboard	https://kubernetes.github.io/dashboard/          
ingress-nginx       	https://kubernetes.github.io/ingress-nginx       
release "ingress-nginx" uninstalled
Release "ingress-nginx" does not exist. Installing it now.
NAME: ingress-nginx
LAST DEPLOYED: Mon Jan 27 09:09:38 2025
NAMESPACE: ingress-nginx
STATUS: deployed
REVISION: 1
TEST SUITE: None
NOTES:
The ingress-nginx controller has been installed.


## Deploy two Delopyments

- The first deployment:
  - Is called `web` and has `replicas` set to 1.
  - Has a Pod template with a container:
    - Listening on `containerPort` 8080
    - Based on the image `gcr.io/google-samples/hello-app:1.0`.
  - The image hosts a web server with a default web page that returns:
    - `Hello, World!`
    - `Version: 1.0.0`
- The second deployment:
  - Is called `web2` and has `replicas` set to 1.
  - Has a Pod template with a container:
    - Listening on `containerPort` 8080
    - Based on the image `gcr.io/google-samples/hello-app:2.0`.
  - The image hosts a web server with a default web page that returns:
    - `Hello, World!`
    - `Version: 2.0.0`
- Both images are pulled from Google's container registry `gcr.io`.

In [3]:
!kubectl create deployment web --image=gcr.io/google-samples/hello-app:1.0
!kubectl create deployment web2 --image=gcr.io/google-samples/hello-app:2.0

deployment.apps/web created
deployment.apps/web2 created


## List Deployments

- We see both Deployments running.
  - The `web` Deployment is using image `gcr.io/google-samples/hello-app:1.0`.
  - The `web2` Deployment is using image `gcr.io/google-samples/hello-app:2.0`.

In [5]:
#!kubectl get deploy -o wide
!kubectl get deployments -o wide

NAME   READY   UP-TO-DATE   AVAILABLE   AGE   CONTAINERS   IMAGES                                SELECTOR
web    1/1     1            1           14s   hello-app    gcr.io/google-samples/hello-app:1.0   app=web
web2   1/1     1            1           14s   hello-app    gcr.io/google-samples/hello-app:2.0   app=web2


## Get Pods

- We see that two Pods are running.

In [6]:
#!kubectl get po -o wide
!kubectl get pods -o wide

NAME                   READY   STATUS    RESTARTS   AGE   IP          NODE             NOMINATED NODE   READINESS GATES
web-56bb54ff6d-7l4sg   1/1     Running   0          23s   10.1.1.34   docker-desktop   <none>           <none>
web2-d8dcbdf99-9htgw   1/1     Running   0          23s   10.1.1.35   docker-desktop   <none>           <none>


## Deploy a ClusterIP Service for each Deployment

- Each Service:
  - Is of type `ClusterIP`.
  - Listens on `port` 8080.
  - Redirects traffic to `targetPort` 8080.

In [7]:
!kubectl expose deployment web --type ClusterIP --port 8080 --target-port 8080
!kubectl expose deployment web2 --type ClusterIP --port 8080 --target-port 8080

service/web exposed
service/web2 exposed


## List Services

- We see the two `ClusterIP` Services for `web` and `web2`.

In [8]:
#!kubectl get svc -o wide
!kubectl get services -o wide

NAME         TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE   SELECTOR
kubernetes   ClusterIP   10.96.0.1       <none>        443/TCP    25d   <none>
web          ClusterIP   10.100.112.58   <none>        8080/TCP   9s    app=web
web2         ClusterIP   10.102.205.69   <none>        8080/TCP   9s    app=web2


## Deploy an Ingress

- The Ingress' definition is in the YAML file `manifests/ingress.yaml`.

In [35]:
!kubectl apply -f manifests/ingress.yaml

ingress.networking.k8s.io/example-ingress configured


## Let's look at the Ingress' YAML

```bash
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: example-ingress                               # the ingress' name
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /$1   # this rewrites the target so that it is prefixed with a slash /
spec:
  ingressClassName: nginx    # this is the ingress class name (must be explicitly set for Docker Desktop)
  rules:
  - host: hello-world.info   # the host determines which domain the ingress triggers on ("Host" in the request header)
    http:
      paths:

      - path: /                # this path is taken if the URL has the default path as a prefix
        pathType: Prefix
        backend:
          service:
            name: web          # the name of the ClusterIP service to redirect to  (the "web" service in this case)
            port:
              number: 8080     # the port to redirect the traffic to (8080)
      
      - path: /v2                   # this path is taken if the URL has the path /v2 as a prefix
        pathType: Prefix
        backend:
          service:
            name: web2              # the name of the ClusterIP service to redirect to  (the "web2" service in this case)
            port:
              number: 8080          # the port to redirect the traffic to (8080)
```

In [36]:
!type manifests\ingress.yaml
#!cat manifests/ingress.yaml # use this on Linux/Mac

apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  name: example-ingress
  annotations:
    nginx.ingress.kubernetes.io/rewrite-target: /$1
spec:
  ingressClassName: nginx
  rules:
  - host: hello-world.info
    http:
      paths:
      - path: /
        pathType: Prefix
        backend:
          service:
            name: web
            port:
              number: 8080
      - path: /v2
        pathType: Prefix
        backend:
          service:
            name: web2
            port:
              number: 8080


## List Ingresses

- We see that the Ingress triggers on
  - Host `hello-world.info` (`HOSTS`).
  - Port 80 (`PORTS`).

In [37]:
!kubectl get ingress

NAME              CLASS   HOSTS              ADDRESS   PORTS   AGE
example-ingress   nginx   hello-world.info             80      35m


## Access the first Application via the Ingress

- The Ingres redirects the traffic to the first application if:
  - The `Host` header is `hello-world.info`.
  - The `path` is the deault path (`/`).
- The first application is based on the image `gcr.io/google-samples/hello-app:1.0`.

In [41]:
!curl -H "Host: hello-world.info" "http://localhost"

Hello, world!
Version: 1.0.0
Hostname: web-56bb54ff6d-7l4sg


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    60  100    60    0     0   2862      0 --:--:-- --:--:-- --:--:--  3000


## Access the second Application via the Ingress

- The Ingres redirects the traffic to the econdsecond application if:
  - The `Host` header is `hello-world.info`.
  - The `path` is (`/v2`).
- The first application is based on the image `gcr.io/google-samples/hello-app:2.0`.

In [42]:
!curl -H "Host: hello-world.info" "http://localhost/v2"

Hello, world!

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    60  100    60    0     0   8079      0 --:--:-- --:--:-- --:--:--  8571



Version: 2.0.0
Hostname: web2-d8dcbdf99-9htgw


## Delete the Ingress, Services and Deployments

In [43]:
!kubectl delete -f manifests/ingress.yaml
!kubectl delete service web
!kubectl delete service web2
!kubectl delete deployment web
!kubectl delete deployment web2

ingress.networking.k8s.io "example-ingress" deleted
service "web" deleted
service "web2" deleted
deployment.apps "web" deleted
deployment.apps "web2" deleted


## List Ingresses, Services, Deployments and Pods

- We see that the Ingress, ClusterIP Services and Deployments with associated Pods have been deleted.

In [44]:
!kubectl get ingress
!kubectl get services
!kubectl get deployments
!kubectl get pods

No resources found in default namespace.


NAME         TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE
kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   25d


No resources found in default namespace.
No resources found in default namespace.
